# Thermal Transfer Learning: Dataset 2 to Dataset 1

This experiment starts from the completed Dataset 2-only detector and fine-tunes it on Dataset 1 images. Its purpose is to improve recognition of Dataset 1's different infrared scenes, target scales, cameras, and annotation geometry while preserving a direct record of what happens to Dataset 2 performance.

**Comparison scope:** initialize from the saved Dataset 2 baseline, then report Dataset 1 and Dataset 2 held-out results separately. This isolates the effect of sequential fine-tuning.

The input size remains **640** so that the changed factor is transfer learning, not resolution. Dataset 1 is used only for fine-tuning and model selection; Dataset 2 remains unseen during fine-tuning. The final checkpoint is evaluated separately on both fixed source-specific test partitions.

## Experiment contract

- **Initialization:** `experiment_a_dataset2_only_batch32/weights/best.pt`.
- **Fine-tuning data:** Dataset 1 `train` partition only.
- **Model selection:** a deterministic 50/50 source-balanced mix of existing Dataset 1 and Dataset 2 validation images.
- **Final tests:** Dataset 1 `test` and Dataset 2-only `test`, each reported separately.
- **Fixed image size:** 640 pixels.
- **Learning rate:** lower than initial training (`1e-4`) to adapt the Dataset 2 features without aggressively overwriting them.
- **Decision threshold:** fixed at `0.25`, the threshold selected on Dataset 2 validation data for the original Dataset 2-only baseline. It is applied unchanged to both held-out tests. Dataset 1 backgrounds are not birds.

This is sequential fine-tuning, not the later source-balanced combined-training experiment. A decline on Dataset 2—particularly a higher bird false-positive rate—would be evidence of catastrophic forgetting.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

WORKING_DIRECTORY = Path.cwd().resolve()
if (WORKING_DIRECTORY / '.git').is_dir():
    REPO_ROOT = WORKING_DIRECTORY
elif (WORKING_DIRECTORY.parent / '.git').is_dir():
    REPO_ROOT = WORKING_DIRECTORY.parent
else:
    raise RuntimeError('Open this notebook from the repository root or notebooks/ directory.')

THERMAL_ROOT = REPO_ROOT / 'data' / 'thermal-ir'
DATASET1_ROOT = THERMAL_ROOT / 'processed' / 'dataset1_only_yolo_v1'
DATASET2_ROOT = THERMAL_ROOT / 'processed' / 'yolo_v1'
DATASET1_YAML = DATASET1_ROOT / 'experiment_c_dataset1_only.yaml'
DATASET2_YAML = DATASET2_ROOT / 'experiment_a_dataset2_only.yaml'
DATASET2_INITIAL_WEIGHTS = REPO_ROOT / 'results' / 'thermal-ir' / 'runs' / 'experiment_a_dataset2_only_batch32' / 'weights' / 'best.pt'
TRANSFER_CONFIG_ROOT = THERMAL_ROOT / 'processed' / 'transfer_learning_v1'
TRANSFER_YAML = TRANSFER_CONFIG_ROOT / 'dataset1_train_source_balanced_validation.yaml'
# v2 disables high-rate warmup so Dataset 2 features are adapted rather than overwritten.
EXPERIMENT_NAME = 'experiment_d_dataset2_to_dataset1_transfer_v2'
RUN_NAME = f'{EXPERIMENT_NAME}_batch32'
RUN_ROOT = REPO_ROOT / 'results' / 'thermal-ir' / 'runs'
EVALUATION_ROOT = REPO_ROOT / 'results' / 'thermal-ir' / 'evaluation' / RUN_NAME

RANDOM_SEED = 590
IMG_SIZE = 640
BATCH_SIZE = 32
INFERENCE_BATCH_SIZE = 8  # Conservative setting for a 16 GiB GPU; increase only if memory permits.
INFERENCE_CONFIDENCE_FLOOR = 0.001
# Fixed from the Dataset 2-only baseline's Dataset 2 validation procedure.
DECISION_THRESHOLD = 0.25

print(f'Python: {sys.version.split()[0]}')
print(f'Dataset 2 starting weights: {DATASET2_INITIAL_WEIGHTS}')
print(f'Dataset 1 fine-tuning YAML: {DATASET1_YAML}')


Python: 3.11.7
Dataset 2 starting weights: D:\Capstone\aai-590-capstone\results\thermal-ir\runs\experiment_a_dataset2_only_batch32\weights\best.pt
Dataset 1 fine-tuning YAML: D:\Capstone\aai-590-capstone\data\thermal-ir\processed\dataset1_only_yolo_v1\experiment_c_dataset1_only.yaml


In [2]:
required = [
    DATASET2_INITIAL_WEIGHTS, DATASET1_YAML, DATASET2_YAML,
    DATASET1_ROOT / 'metadata' / 'frames.csv',
    DATASET2_ROOT / 'metadata' / 'frames.csv',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Transfer-learning prerequisites are missing: ' + ', '.join(missing))
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Shut down other GPU-using notebook kernels, then restart this kernel.')

dataset1_frames = pd.read_csv(DATASET1_ROOT / 'metadata' / 'frames.csv')
dataset2_frames = pd.read_csv(DATASET2_ROOT / 'metadata' / 'frames.csv')
dataset1_validation = dataset1_frames.loc[dataset1_frames.split.eq('val')].copy()
dataset2_validation = dataset2_frames.loc[(dataset2_frames.source_dataset == 'drone_detection_dataset2') & dataset2_frames.split.eq('val')].copy()
if dataset1_validation.empty or dataset2_validation.empty:
    raise RuntimeError('Both source-specific validation partitions must contain images.')
# Ultralytics reports one validation loss/fitness value. Balance sources so Dataset 1's
# larger validation set does not dominate that training-time signal.
per_source_validation_count = min(len(dataset1_validation), len(dataset2_validation))
balanced_dataset1_validation = dataset1_validation.sample(n=per_source_validation_count, random_state=RANDOM_SEED)
balanced_validation = pd.concat([balanced_dataset1_validation, dataset2_validation], ignore_index=True).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
TRANSFER_CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
validation_list = TRANSFER_CONFIG_ROOT / 'source_balanced_validation_images.txt'
validation_list.write_text('\n'.join(str((REPO_ROOT / image_path).resolve()) for image_path in balanced_validation.image_path) + '\n', encoding='utf-8')
transfer_data = {
    'path': str(DATASET1_ROOT),
    'train': 'images/train',
    'val': str(validation_list.resolve()),
    'test': 'images/test',
    'names': {0: 'drone'},
}
with TRANSFER_YAML.open('w', encoding='utf-8') as stream:
    yaml.safe_dump(transfer_data, stream, sort_keys=False)
display(dataset1_frames.groupby(['split', 'export_policy']).size().rename('Dataset 1 images').to_frame())
display(dataset2_frames.loc[dataset2_frames.source_dataset.eq('drone_detection_dataset2')].groupby(['split', 'export_policy']).size().rename('Dataset 2 images').to_frame())
print(f'Balanced validation images: {len(balanced_validation):,} ({per_source_validation_count:,} from each source)')
print(f'GPU: {torch.cuda.get_device_name(0)}')


Dataset 1 images
split export_policy                           
test  negative_target_absent               610
      positive_drone                      6632
train negative_target_absent              2499
      positive_drone                     31822
val   negative_target_absent               414
      positive_drone                      7436

Dataset 2 images
split export_policy                   
test  negative_bird                753
      positive_drone              1878
train negative_bird               3369
      positive_drone              6138
val   negative_bird                741
      positive_drone              1430

Balanced validation images: 4,342 (2,171 from each source)
GPU: NVIDIA GeForce RTX 5080


## Fine-tune the Dataset 2 checkpoint on Dataset 1

The optimizer state is reset. The detector weights are loaded from Dataset 2 `best.pt`, and a source-balanced mix of the existing Dataset 1 and Dataset 2 validation partitions controls early stopping. No Dataset 2 images appear in this training stage. Ultralytics reports one pooled validation loss/fitness value; the equal source counts prevent Dataset 1 from dominating it.

In [3]:
TRANSFER_CONFIG = {
    'imgsz': IMG_SIZE,
    'epochs': 50,
    'batch': BATCH_SIZE,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 1e-4,
    # The default warmup_bias_lr=0.1 overwhelms a 1e-4 transfer-learning rate.
    'warmup_epochs': 0.0,
    'warmup_bias_lr': 1e-4,
    'weight_decay': 5e-4,
    'patience': 15,
    'seed': RANDOM_SEED,
    'deterministic': True,
    'workers': 8,
    'pretrained': True,
    'resume': False,
    'cache': False,
    'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0,
    'fliplr': 0.5, 'flipud': 0.0,
    'degrees': 0.0, 'translate': 0.10, 'scale': 0.25,
    'mosaic': 0.5, 'close_mosaic': 10,
}

def fine_tune_dataset2_on_dataset1() -> Path:
    run_directory = RUN_ROOT / RUN_NAME
    if run_directory.exists():
        raise RuntimeError(
            f'{run_directory} already exists. Use its saved best.pt for evaluation or choose a new run name; do not overwrite an experiment.'
        )
    model = YOLO(str(DATASET2_INITIAL_WEIGHTS))
    model.train(
        data=str(TRANSFER_YAML),
        project=str(RUN_ROOT),
        name=RUN_NAME,
        exist_ok=False,
        **TRANSFER_CONFIG,
    )
    return Path(model.trainer.save_dir)


In [4]:
# Run once after shutting down any other GPU-using notebook kernels.
training_run_dir = fine_tune_dataset2_on_dataset1()
BEST_WEIGHTS = training_run_dir / 'weights' / 'best.pt'
if not BEST_WEIGHTS.is_file():
    raise RuntimeError(f'Expected best checkpoint was not written: {BEST_WEIGHTS}')
print(f'Transfer-learning checkpoint: {BEST_WEIGHTS}')


New https://pypi.org/project/ultralytics/8.4.213 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.198  Python-3.11.7 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5080, 16276MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Capstone\aai-590-capstone\data\thermal-ir\processed\transfer_learning_v1\dataset1_train_source_balanced_validation.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_widt

## Apply the fixed Dataset 2 decision threshold

This transfer experiment retains the Dataset 2-only baseline's validation-selected threshold of **0.25**. No Dataset 1 validation inference or threshold tuning is performed after training. The code below only loads the saved transfer checkpoint and defines memory-safe, explicitly chunked inference for the held-out tests.

In [4]:
def predict_image_scores(model: YOLO, rows: pd.DataFrame) -> pd.DataFrame:
    # A Python list of all paths is treated as one giant batch by Ultralytics.
    # Submit explicit small chunks so inference fits a 16 GiB GPU.
    records = rows.to_dict('records')
    scored = []
    for start in range(0, len(records), INFERENCE_BATCH_SIZE):
        batch_records = records[start:start + INFERENCE_BATCH_SIZE]
        batch_paths = [str((REPO_ROOT / row['image_path']).resolve()) for row in batch_records]
        predictions = list(model.predict(
            source=batch_paths, stream=True, conf=INFERENCE_CONFIDENCE_FLOOR,
            device=0, verbose=False,
        ))
        if len(predictions) != len(batch_records):
            raise RuntimeError(f'Expected {len(batch_records)} predictions but received {len(predictions)}.')
        for row, prediction in zip(batch_records, predictions):
            confidences = [float(value) for value in prediction.boxes.conf]
            scored.append({
                **row,
                'ground_truth_drone_present': row['label_status'] == 'drone_positive',
                'max_confidence': max(confidences, default=0.0),
                'prediction_count_at_0_001': len(confidences),
            })
        del predictions
        torch.cuda.empty_cache()
    return pd.DataFrame(scored)

def threshold_metrics(scores: pd.DataFrame, threshold: float) -> tuple[pd.DataFrame, dict]:
    evaluated = scores.copy()
    evaluated['predicted_drone_present'] = evaluated.max_confidence.ge(threshold)
    evaluated['binary_outcome'] = np.select(
        [
            evaluated.ground_truth_drone_present & evaluated.predicted_drone_present,
            ~evaluated.ground_truth_drone_present & evaluated.predicted_drone_present,
            evaluated.ground_truth_drone_present & ~evaluated.predicted_drone_present,
        ],
        ['true_positive', 'false_positive', 'false_negative'], default='true_negative',
    )
    counts = evaluated.binary_outcome.value_counts()
    tp, fp = int(counts.get('true_positive', 0)), int(counts.get('false_positive', 0))
    fn, tn = int(counts.get('false_negative', 0)), int(counts.get('true_negative', 0))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return evaluated, {
        'decision_threshold': threshold, 'true_positive': tp, 'false_positive': fp,
        'false_negative': fn, 'true_negative': tn, 'precision': precision, 'recall': recall,
        'f1': 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        'target_absent_false_positive_rate': fp / (fp + tn) if fp + tn else 0.0,
    }

# This lets evaluation run in a fresh kernel without rerunning the two-hour training cell.
BEST_WEIGHTS = RUN_ROOT / RUN_NAME / 'weights' / 'best.pt'
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f'Transfer-learning checkpoint is missing: {BEST_WEIGHTS}')
torch.cuda.empty_cache()
model = YOLO(str(BEST_WEIGHTS))
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Loaded saved transfer checkpoint: {BEST_WEIGHTS}')
print(f'Fixed Dataset 2 baseline threshold: {DECISION_THRESHOLD:.2f}')


Loaded saved transfer checkpoint: D:\Capstone\aai-590-capstone\results\thermal-ir\runs\experiment_d_dataset2_to_dataset1_transfer_v2_batch32\weights\best.pt
Fixed Dataset 2 baseline threshold: 0.25


## Evaluate each held-out source separately

The checkpoint and threshold are now locked. The following cell produces separate detection metrics and image-level decisions for Dataset 1 and Dataset 2 test partitions. It does not pool their metrics or relabel Dataset 1 backgrounds as birds.

In [5]:
def evaluate_source_test(source_name: str, data_yaml: Path, metadata: pd.DataFrame) -> dict:
    source_rows = metadata.loc[metadata.split.eq('test')].copy().reset_index(drop=True)
    source_root = EVALUATION_ROOT / source_name
    source_root.mkdir(parents=True, exist_ok=True)
    detection = model.val(
        data=str(data_yaml), split='test', conf=INFERENCE_CONFIDENCE_FLOOR,
        batch=INFERENCE_BATCH_SIZE, device=0, plots=True,
        project=str(source_root), name='ultralytics_detection', exist_ok=True,
    )
    (source_root / 'test_detection_metrics.json').write_text(json.dumps(detection.results_dict, indent=2), encoding='utf-8')
    # Release evaluation tensors before batched inference on the same GPU.
    del detection
    torch.cuda.empty_cache()
    scores = predict_image_scores(model, source_rows)
    decisions, summary = threshold_metrics(scores, DECISION_THRESHOLD)
    if source_name == 'dataset2_test':
        bird_rows = decisions.export_policy.eq('negative_bird')
        summary['bird_false_positive_rate'] = float(decisions.loc[bird_rows, 'predicted_drone_present'].mean())
    else:
        summary['limitation'] = 'Dataset 1 target-absent frames are background negatives, not birds.'
    summary['decision_threshold_origin'] = 'fixed_dataset2_baseline_validation_threshold'
    scores.to_csv(source_root / 'test_image_scores.csv', index=False)
    decisions.to_csv(source_root / 'test_image_decisions.csv', index=False)
    (source_root / 'test_decision_metrics.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return {'source': source_name, **detection_results_if_available(source_root), **summary}

def detection_results_if_available(source_root: Path) -> dict:
    return json.loads((source_root / 'test_detection_metrics.json').read_text(encoding='utf-8'))

dataset2_only_frames = dataset2_frames.loc[dataset2_frames.source_dataset.eq('drone_detection_dataset2')].copy()
dataset1_summary = evaluate_source_test('dataset1_test', DATASET1_YAML, dataset1_frames)
torch.cuda.empty_cache()
dataset2_summary = evaluate_source_test('dataset2_test', DATASET2_YAML, dataset2_only_frames)
comparison = pd.DataFrame([dataset1_summary, dataset2_summary]).set_index('source')
comparison.to_csv(EVALUATION_ROOT / 'source_specific_test_comparison.csv')
display(comparison.T)


Ultralytics 8.4.198  Python-3.11.7 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5080, 16276MiB)
YOLO26n summary (fused): 121 layers, 2,389,735 parameters, 0 gradients, 5.4 GFLOPs
WARNING val: Slow image access detected (ping: 0.3±0.1 ms, read: 12.3±9.3 MB/s, size: 52.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\Capstone\aai-590-capstone\data\thermal-ir\processed\dataset1_only_yolo_v1\labels\test.cache... 7242 images, 610 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7242/7242  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 906/906 20.3it/s 44.5s0.1ss
                   all       7242       6632      0.908      0.709      0.752       0.39
Speed: 0.5ms preprocess, 1.3ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to D:\Capstone\aai-590-capstone\results\thermal-ir\evaluation\experiment_d_datas

source,dataset1_test,dataset2_test
metrics/precision(B),0.908473,0.269648
metrics/recall(B),0.70941,0.198616
metrics/mAP50(B),0.752194,0.095954
metrics/mAP50-95(B),0.38952,0.018889
fitness,0.38952,0.018889
decision_threshold,0.25,0.25
true_positive,5332,1818
false_positive,36,409
false_negative,1300,60
true_negative,574,344


## Interpretation

The desired outcome is higher Dataset 1 recall than the Dataset 2-only model achieved on Dataset 1, while retaining enough Dataset 2 recall and bird rejection for the project goal. If Dataset 1 improves but Dataset 2 bird false positives rise substantially, report the result as source adaptation with catastrophic-forgetting risk—not as a universally improved detector.

Do not use either test table to select another checkpoint or threshold. A later source-balanced combined experiment can use its own validation partitions to optimize the tradeoff.